# SQL marts, segmentation logic, and point-in-time loan features

**Research question:** How can historical ledger data support customer diagnosis without overstating modern fintech or credit evidence?

This notebook is the review interface. Reusable transformations remain in `src/`, `scripts/`, and `sql/`; the notebook reads their committed evidence rather than creating a second implementation. Raw source data is intentionally not embedded.

## Hypotheses declared before interpretation

- Opening deposits and system postings inflate naive activation.
- Rule segments should remain descriptive and fully traceable.
- Loan features must stop strictly before origination.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(f'Repository: {ROOT.name}')


Repository: novapay-fintech-analytics


## Feature provenance

Every row states its raw source, transformation, decision-time cutoff, fit scope, missing-value treatment, and known leakage or proxy risk.

In [2]:
features = pd.read_csv(ROOT / 'reports' / 'feature_provenance.csv')
display(features)
display(features.groupby(['feature_family', 'model_role']).size().rename('features').reset_index())


,feature_name,feature_family,raw_source,grain,transformation,observation_cutoff,available_at_decision,fit_partition,missing_treatment,model_role,exclusion_reason,leakage_risk,proxy_or_fairness_risk,validation
0,loan_amount_czk,loan terms,loans.loan_amount_czk,loan,Raw origination amount,loan_date,yes,none,median,model,NaN,low,credit amount may reflect prior underwriting,temporal OOT
1,duration_months,loan terms,loans.duration_months,loan,Raw term,loan_date,yes,none,median,model,NaN,low,may encode policy,temporal OOT
2,monthly_payment_czk,loan terms,loans.monthly_payment_czk,loan,Raw scheduled payment,loan_date,yes,none,median,model,NaN,low,affordability proxy,temporal OOT
3,account_tenure_days,relationship,accounts.opened_date,loan,loan_date - opened_date,loan_date,yes,none,median,model,NaN,low,tenure may proxy customer age/access,point-in-time contract
4,prior_transaction_count,past behaviour,transactions,loan,Count transactions where transaction_date < lo...,strictly before loan_date,yes,none,median,model,NaN,low,relationship-depth proxy,point-in-time contract
5,prior_credit_czk,past behaviour,transactions,loan,Sum prior credits,strictly before loan_date,yes,none,zero,model,NaN,low,income proxy,point-in-time contract
6,prior_debit_czk,past behaviour,transactions,loan,Sum prior debits,strictly before loan_date,yes,none,zero,model,NaN,low,spend proxy,point-in-time contract
7,prior_net_flow_czk,past behaviour,transactions,loan,Prior credits minus prior debits,strictly before loan_date,yes,none,zero,model,NaN,low,income/spend proxy,point-in-time contract
8,prior_latest_balance_czk,past balance,transactions.balance_czk,loan,Balance from latest strictly prior transaction,strictly before loan_date,yes,none,median,model,NaN,low,wealth proxy,point-in-time contract
9,prior_min_balance_czk,past balance,transactions.balance_czk,loan,Minimum strictly prior balance,strictly before loan_date,yes,none,median,model,NaN,low,financial distress proxy,point-in-time contract


,feature_family,model_role,features
0,account,model,1
1,district context,model,3
2,loan terms,model,3
3,past balance,model,2
4,past behaviour,model,5
5,relationship,model,1


## Decision-time audit

A feature is eligible only when its information exists at the stated observation cutoff. Model-fitted bins, encoders, imputers, and thresholds use development data only. Outcomes and evaluation groups never enter the predictor matrix.

In [3]:
audit = features[['feature_name', 'observation_cutoff', 'available_at_decision', 'fit_partition', 'leakage_risk', 'validation']]
display(audit)


,feature_name,observation_cutoff,available_at_decision,fit_partition,leakage_risk,validation
0,loan_amount_czk,loan_date,yes,none,low,temporal OOT
1,duration_months,loan_date,yes,none,low,temporal OOT
2,monthly_payment_czk,loan_date,yes,none,low,temporal OOT
3,account_tenure_days,loan_date,yes,none,low,point-in-time contract
4,prior_transaction_count,strictly before loan_date,yes,none,low,point-in-time contract
5,prior_credit_czk,strictly before loan_date,yes,none,low,point-in-time contract
6,prior_debit_czk,strictly before loan_date,yes,none,low,point-in-time contract
7,prior_net_flow_czk,strictly before loan_date,yes,none,low,point-in-time contract
8,prior_latest_balance_czk,strictly before loan_date,yes,none,low,point-in-time contract
9,prior_min_balance_czk,strictly before loan_date,yes,none,low,point-in-time contract


## Committed analytical evidence

In [4]:
evidence_files = ["dashboard/data/segment_summary.csv", "reports/loan_risk_temporal_metrics.csv", "reports/loan_risk_temporal_validation.json"]
for relative in evidence_files:
    path = ROOT / relative
    display(Markdown(f'### `{relative}`'))
    if not path.exists():
        display(Markdown('*Not committed; reproduce locally using the README pipeline.*'))
    elif path.suffix == '.csv':
        display(pd.read_csv(path).head(20))
    elif path.suffix == '.json':
        payload = json.loads(path.read_text())
        display(payload)


### `dashboard/data/segment_summary.csv`

,customer_segment,accounts,portfolio_share,average_days_to_activation,average_engaged_transactions,average_latest_balance_czk,negative_balance_accounts,problem_loan_accounts,segment_definition,recommended_action
0,Light relationship,1964,0.4364,53.2,81.5,39589.50,0,0,Below-median engagement and no higher-priority...,Use onboarding education and low-friction acti...
1,Core transactional,1357,0.3016,40.3,225.6,37826.80,0,0,Above-median engagement without a card or loan...,Promote card adoption and additional account f...
2,Healthy borrower,525,0.1167,45.8,174.6,47966.10,0,0,Completed-good or active-good loan relationship,Offer relevant credit servicing and repayment ...
3,Card-led engaged,275,0.0611,49.2,211.0,64182.19,0,0,Cardholder with above-median engagement,Deepen card usage with merchant and recurring-...
4,High-value relationship,274,0.0609,42.3,301.9,83268.78,0,0,Top-quartile credit value ending balance and e...,Protect service quality and test premium relat...
5,Risk watch,105,0.0233,57.7,169.2,22928.85,39,76,Negative balance or observed problem-loan status,Prioritise review repayment support and balanc...


### `reports/loan_risk_temporal_metrics.csv`

,partition,rows,positive_labels,problem_rate,average_precision,roc_auc,brier_score
0,development,399,54,0.135338,0.518284,0.844444,0.092848
1,calibration,142,19,0.133803,0.819140,0.922122,0.052546
2,oot,141,3,0.021277,0.618056,0.966184,0.027234


### `reports/loan_risk_temporal_validation.json`

{'label': 'source status B or D; historical observed outcome, not regulatory default',
 'development_end': '1997-05-22',
 'calibration_end': '1998-01-23',
 'features': ['loan_amount_czk',
  'duration_months',
  'monthly_payment_czk',
  'account_tenure_days',
  'prior_transaction_count',
  'prior_credit_czk',
  'prior_debit_czk',
  'prior_net_flow_czk',
  'prior_latest_balance_czk',
  'prior_min_balance_czk',
  'prior_active_days',
  'average_salary_czk',
  'unemployment_rate_1996',
  'statement_frequency',
  'region_name'],
 'probability_calibration': 'Platt scaling fitted only on the later calibration window',
 'metrics': [{'partition': 'development',
   'rows': 399,
   'positive_labels': 54,
   'problem_rate': 0.13533834586466165,
   'average_precision': 0.5182839602237351,
   'roc_auc': 0.8444444444444444,
   'brier_score': 0.09284779944334952},
  {'partition': 'calibration',
   'rows': 142,
   'positive_labels': 19,
   'problem_rate': 0.13380281690140844,
   'average_precision': 0.

## Decision

**Use segments for transparent service hypotheses and keep the loan model research-only because locked OOT contains three positive labels.**

### Claim boundary

The notebook distinguishes observed evidence from model output, simulated policy, and causal impact. A strong offline metric is not production exposure.